# Tank 0.1 — `tank check` demo

**The thesis in one sentence:** the project declares its ontology in code, and Tank
verifies — deterministically, with no LLM involved — that the SurrealDB database
respects the declaration. If it doesn't, the build breaks.

What this demo shows:
1. A **minimal** ontology and a passing check (golden)
2. The **sabotages**: wrong table, wrong field, vocabulary drift, broken ontology
3. The **full** ontology (a news graph: relations + vector + full-text + freshness)
4. The state nobody else reports: **VACUOUS** (an empty table never passes silently)
5. How this runs in CI (exit codes, JSON)

> Check-code reference (`TBL-001`, `VEC-002`…): [`docs/checks.md`](../docs/checks.md)

In [1]:
# Setup: local SurrealDB v3 connection and demo seeds
import base64, json, urllib.request
from pathlib import Path

from tank import (
    Attr,
    FullText,
    Locator,
    Ontology,
    OntologyError,
    Relation,
    Scope,
    StableId,
    UnitType,
    Vector,
    Weight,
    Freshness,
)
from tank.checks import run_check

URL, USER, PASSWORD, NS = "http://127.0.0.1:8019", "root", "root", "tank_demo"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())


def sql(db, statements, allow_errors=False):
    auth = base64.b64encode(f"{USER}:{PASSWORD}".encode()).decode()
    req = urllib.request.Request(
        f"{URL}/sql",
        data=statements.encode(),
        headers={
            "Authorization": f"Basic {auth}",
            "Accept": "application/json",
            "surreal-ns": NS,
            "surreal-db": db,
        },
    )
    with urllib.request.urlopen(req, timeout=20) as r:
        results = json.load(r)
    errors = [x for x in results if x.get("status") != "OK"]
    assert allow_errors or not errors, errors[:2]
    return results


def fresh_db(db, fixture=None):
    sql(db, f"REMOVE DATABASE IF EXISTS `{db}`;", allow_errors=True)
    sql(
        db, "DEFINE PARAM $bootstrap VALUE 1;", allow_errors=True
    )  # ns/db only materialize on a write
    sql(db, "DEFINE PARAM $bootstrap2 VALUE 1;")
    if fixture:
        sql(db, (ROOT / "tests" / "fixtures" / fixture / "seed.surql").read_text())


async def check(ontology, db):
    return await run_check(
        ontology, url=URL, namespace=NS, database=db, user=USER, password=PASSWORD
    )


def problems_only(report):
    "Compact view: everything that is not PASS."
    for f in report.findings:
        if f.status != "PASS":
            print(f"  {f}")
    print(f"\n  exit code: {report.exit_code()}")


fresh_db("demo_minimal", "minimal")
fresh_db("demo_full", "news_mini")
print("Seeds applied at", URL, "· ns", NS)

Seeds applied at http://127.0.0.1:8019 · ns tank_demo


## 1. The minimal ontology

One domain type (`report`), mapped to the table where it lives (`technical_assessment` —
different names **on purpose**: an agent has to read the declaration, not guess).
The `attrs` declare the queryable fields: what an agent may use in a `WHERE`/`ORDER BY`,
with a closed value vocabulary.

In [2]:
minimal = Ontology(
    types=[
        UnitType(
            "report",
            table="technical_assessment",
            nature="original",
            id=StableId.of("code"),
            text="body_text",
            locator=Locator(source="code"),
            attrs=[
                Attr("status", "string", values=["current", "revoked"]),
                Attr("issued_at", "datetime"),
            ],
        ),
    ],
)
print(minimal.to_json()[:600], "…")  # the JSON export: the surface a skill hands to an agent

{
  "types": [
    {
      "name": "report",
      "table": "technical_assessment",
      "nature": "original",
      "id": {
        "fields": [
          "code"
        ],
        "version_fields": []
      },
      "text": "body_text",
      "attrs": [
        {
          "name": "status",
          "type": "string",
          "values": [
            "current",
            "revoked"
          ],
          "description": null
        },
        {
          "name": "issued_at",
          "type": "datetime",
          "values": null,
          "description": null
        }
      ],
      "loca …


## 2. The golden check — database and ontology agree

In [3]:
report = await check(minimal, "demo_minimal")
print(report.render_text())

tank check
  server:    http://127.0.0.1:8019 (surrealdb-3.1.6+20260813.cfbaec4)
  target:    ns=tank_demo db=demo_minimal
  tables:    technical_assessment=12

  PASS    TBL-001  type:report: table 'technical_assessment' exists (SCHEMAFULL, kind=normal)
  PASS    FLD-001  type:report: field 'body_text' is defined on 'technical_assessment'
  PASS    FLD-001  type:report: field 'code' is defined on 'technical_assessment'
  PASS    FLD-001  type:report: attr 'issued_at': declared 'datetime' matches DEFINE FIELD type 'datetime'
  PASS    FLD-001  type:report: attr 'status': declared 'string' matches DEFINE FIELD type 'string'
  PASS    ATTR-010 type:report: attr 'status': sampled values within declared vocabulary

  6 pass, 0 fail, 0 warn, 0 vacuous

  A PASS here means: the declaration is not contradicted by the schema
  and the sampled data of THIS environment. Not verified:
    - semantics of names — whether a declared type/relation means what you think it means
    - embedding model i

## 3. Sabotage: the declared table does not exist

The canonical error: the ontology says *"person supports team"* — and the `supports`
table is not there. *"You are giving me an ontology that does not exist in the database."*

In [4]:
sabotaged = minimal.model_copy(deep=True)
sabotaged.types[0].table = "technical_assessment_v2"  # <- does not exist
problems_only(await check(sabotaged, "demo_minimal"))

  FAIL    TBL-001  type:report: type 'report' declares table 'technical_assessment_v2', which does not exist in the database

  exit code: 1


## 4. Sabotage: the text field has a different name

In [5]:
sabotaged = minimal.model_copy(deep=True)
sabotaged.types[0].text = "body"  # <- the real field is body_text
problems_only(await check(sabotaged, "demo_minimal"))

  FAIL    FLD-002  type:report: field 'body' has no DEFINE FIELD on 'technical_assessment' and is absent from every sampled row

  exit code: 1


## 5. Vocabulary drift — WARN, not FAIL

The ontology declares fewer values than the data holds. Not a structural
contradiction, but a suspicion: it becomes a warning (`--strict` promotes it to an error in CI).

In [6]:
narrow = Ontology(
    types=[
        UnitType(
            "report",
            table="technical_assessment",
            text="body_text",
            attrs=[Attr("status", "string", values=["current"])],
        )
    ]
)  # 'revoked' missing
problems_only(await check(narrow, "demo_minimal"))

  WARN    ATTR-010 type:report: attr 'status': sampled values ['revoked'] are outside the declared vocabulary ['current']

  exit code: 0


## 6. Internally broken ontology — the build breaks with NO database

A cross-reference that does not resolve blows up **at import time**, with every
violation at once. That is "if the ontology breaks, CI breaks" — before a connection exists.

In [7]:
try:
    Ontology(
        types=[UnitType("report", table="technical_assessment")],
        relations=[Relation("issued_by", "report", "engineer")],  # undeclared type
        scopes=[Scope("site", via="belongs_to")],  # undeclared relation
    )
except OntologyError as e:
    print(e)

ontology is invalid (2 violation(s)):
  [ONT-002] relation 'issued_by': to='engineer' is not a declared unit type (declared: ['report'])
  [ONT-003] scope 'site': via='belongs_to' is not a declared relation


## 7. The full ontology — a news graph

5 types (`news`, `entity`, `topic`, `document`, `feed`), 3 edges with declared direction,
a weight on one edge, one field link, vector search (dimension + metric), full-text
(analyzer + language), a scope and freshness. **The index is not declared — it is derived**:
declaring `Vector(dim=4)` makes the check demand the matching HNSW index.

In [8]:
full = Ontology(
    types=[
        UnitType(
            "news",
            table="news",
            nature="original",
            id=StableId.of("title", "published_at"),
            text="body",
            attrs=[Attr("title", "string"), Attr("published_at", "datetime")],
            vector=Vector("emb", 4, "cosine"),
            fulltext=FullText("body", analyzer="az_en", language="english"),
        ),
        UnitType(
            "entity",
            table="entity",
            attrs=[
                Attr("name", "string"),
                Attr("kind", "string", values=["person", "agency", "organization"]),
            ],
        ),
        UnitType("topic", table="topic", attrs=[Attr("name", "string")]),
        UnitType(
            "document",
            table="document",
            attrs=[
                Attr("number", "string"),
                Attr("status", "string", values=["pending", "approved", "rejected"]),
            ],
        ),
        UnitType("feed", table="feed", attrs=[Attr("name", "string")]),
    ],
    relations=[
        Relation("mentions", "news", "entity", weight=Weight("relevance")),
        Relation("about", "news", "topic"),
        Relation("cites", "news", "document"),
        Relation("from_feed", "news", "feed", kind="field_link", field="feed"),
    ],
    scopes=[Scope("entity", via="mentions")],
    freshness=[Freshness("news", "published_at", "30d")],
)
report = await check(full, "demo_full")
print(report.render_text())

tank check
  server:    http://127.0.0.1:8019 (surrealdb-3.1.6+20260813.cfbaec4)
  target:    ns=tank_demo db=demo_full
  tables:    about=2, cites=2, document=2, entity=2, feed=1, mentions=3, news=3, topic=2

  PASS    TBL-001  type:news: table 'news' exists (SCHEMAFULL, kind=normal)
  PASS    FLD-001  type:news: field 'body' is defined on 'news'
  PASS    FLD-001  type:news: field 'emb' is defined on 'news'
  PASS    FLD-001  type:news: attr 'published_at': declared 'datetime' matches DEFINE FIELD type 'datetime'
  PASS    FLD-001  type:news: attr 'title': declared 'string' matches DEFINE FIELD type 'string'
  PASS    VEC-002  type:news: index 'idx_vec' DIMENSION matches (4)
  PASS    VEC-003  type:news: index 'idx_vec' DIST matches (cosine)
  PASS    VEC-010  type:news: sampled embedding dimensions all match 4
  PASS    FTS-001  type:news: FTS index 'idx_fts' covers 'body' (analyzer: az_en)
  PASS    TBL-001  type:entity: table 'entity' exists (SCHEMAFULL, kind=normal)
  PASS    FLD

## 8. Sabotaging the full ontology

**8a. Declared dimension ≠ index.** The ontology says 8, the index has 4 — and the
sampled embeddings are 4-dimensional too (mixed dimensions are a real, silent production
failure mode).

In [9]:
wrong = full.model_copy(deep=True)
wrong.types[0].vector = Vector("emb", 8, "cosine")
problems_only(await check(wrong, "demo_full"))

  FAIL    VEC-002  type:news: index 'idx_vec' has DIMENSION 4, ontology declares dim=8
  FAIL    VEC-010  type:news: sampled embeddings in 'emb' have dimensions [4], expected 8 — rows ingested before the index existed are not defended by the server

  exit code: 1


**8b. Vector index dropped from the database** — the declared search lost its engine.

In [10]:
fresh_db("demo_sab", "news_mini")
sql("demo_sab", "REMOVE INDEX idx_vec ON TABLE news;")
problems_only(await check(full, "demo_sab"))

  FAIL    VEC-001  type:news: vector declared on field 'emb' but no ANN index (HNSW/MTREE/DISKANN) covers it on 'news' — vector search would be a full scan or an error

  exit code: 1


**8c. Edge table dropped** — the same class of error, now on a relation.

In [11]:
fresh_db("demo_sab", "news_mini")
sql("demo_sab", "REMOVE TABLE cites;")
problems_only(await check(full, "demo_sab"))

  FAIL    REL-001  relation:cites: relation 'cites' declares edge table 'cites', which does not exist in the database

  exit code: 1


**8d. An edge created "on the fly"** (just `RELATE`, no `DEFINE TABLE ... TYPE RELATION`).
It works, but nothing defends the direction on write — the check approves by sampling
and **recommends** making it explicit.

In [12]:
fresh_db("demo_sab", "news_mini")
sql("demo_sab", "REMOVE TABLE about; RELATE news:n1->about->topic:t1;")
problems_only(await check(full, "demo_sab"))

  WARN    REL-002  relation:about: edge 'about' endpoints match by sampling, but it has no IN/OUT constraint — nothing defends direction on write; recommend DEFINE TABLE about TYPE RELATION IN news OUT topic

  exit code: 0


## 9. Empty table — VACUOUS, never a silent PASS

A perfect schema with zero rows passes every structural check "by vacuity".
Tank reports that as a state of its own — in CI, `--strict` treats it as a failure.

In [13]:
fresh_db("demo_empty")
ddl = "\n".join(
    l
    for l in (ROOT / "tests/fixtures/minimal/seed.surql").read_text().splitlines()
    if l.startswith("DEFINE")
)
sql("demo_empty", ddl)
report = await check(minimal, "demo_empty")
problems_only(report)
print(f"  exit code with --strict: {report.exit_code(strict=True)}")

  VACUOUS VAC-001  type:report: table 'technical_assessment' has 0 rows — every sampling check for this type is vacuous; structural checks still apply

  exit code: 0
  exit code with --strict: 1


## 10. In CI

```bash
uv run tank check --ontology ontology.py --url $SURREAL_URL --ns prod --db app --strict --json
```

Exit `0` = consistent · `1` = the database contradicts the ontology · `2` = invalid
ontology or connection error. `--json` yields the structured report for machines:

In [14]:
report = await check(minimal, "demo_minimal")
print(json.dumps(json.loads(report.model_dump_json())["findings"][0], indent=2, ensure_ascii=False))

{
  "code": "TBL-001",
  "status": "PASS",
  "subject": "type:report",
  "table": "technical_assessment",
  "message": "table 'technical_assessment' exists (SCHEMAFULL, kind=normal)"
}


---

## What comes next (beyond 0.1)

- **0.2**: the access decorator (`@access_tool`) — usage records and exceptions, the raw material of eval/analytics
- **Generation test**: an agent reads ontology + skill and writes the access function — golden/sabotage/**ablation** with an LLM

Full check-code reference: [`docs/checks.md`](../docs/checks.md)